## Notebook46

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
! pip install igraph --quiet

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs
from funs import DSNetwork

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
case = pl.read_csv(ub + "data/scotus_case.csv")
cite = pl.read_csv(ub + "data/scotus_citation.csv")

### Questions

This notebook continues with the U.S. Supreme Court citation network. Each row of
`case` is one decided case, identified by its `citation`, and each row of `cite` is
one link: the case in `citing_case` cites the earlier case in `cited_case`.

As in the last notebook, we focus on the 80 most-cited cases. The block below is
given. It saves their citations as a list called `top`.

In [ ]:
top = (
    cite
    .group_by(c.cited_case)
    .agg(n_in = pl.len())
    .sort([c.n_in, c.cited_case], descending=[True, False])
    .head(80)
    .get_column("cited_case")
    .to_list()
)

Unless a question says otherwise, just print the result of each block; do not save
it to a variable.

1. Filter `case` to the 80 cases in `top` using `c.citation.is_in(top)`. Select
`case_name` and `term` and sort by `term`.

Only 79 rows print, because one of the 80 citations, `"75 U.S. 168"`, has no row
in `case`. The cases run from *Marbury v. Madison* (1803) to *Chevron* (1983).

2. Filter `cite` to the rows where `citing_case` is `"347 U.S. 483"`
(*Brown v. Board of Education*) and `cited_case` is in `top`. These are the other
top cases that *Brown* cites.

*Brown* cites only two of the other top cases: *ex parte Virginia* (`100 U.S. 339`)
and the *Slaughter-House Cases* (`83 U.S. 36`).

3. Two cases are *co-cited* when a third case cites both of them. To find these
pairs, filter `cite` to the rows where `cited_case` is in `top`, and then join the
result to itself on `citing_case`:

Each row is a case (`citing_case`) together with two of the top cases it cites. The
join also pairs every case with itself, and each pair appears in both orders.

4. Take the code from the previous question and keep each pair once with
`.filter(c.cited_case < c.cited_case_right)`. Then count the rows for each pair of
`cited_case` and `cited_case_right`, sort from most to least common, and keep the
first 10 rows.

The pair cited together most often is `308 U.S. 147` and `310 U.S. 296`
(*Schneider v. State* and *Cantwell v. Connecticut*), 84 times. Second is
*Furman v. Georgia* and *Gregg v. Georgia*, the two death-penalty cases.

5. The block below is given. It keeps the pairs co-cited at least 15 times, builds
an undirected network with `DSNetwork.process()`, and joins the case names onto the
node table. The results are saved as `co_node` and `co_edge`. Run it and look at
`co_node`.

In [ ]:
cocite_edges = (
    cite
    .filter(c.cited_case.is_in(top))
    .join(cite.filter(c.cited_case.is_in(top)), on=c.citing_case)
    .filter(c.cited_case < c.cited_case_right)
    .group_by(c.cited_case, c.cited_case_right)
    .agg(count = pl.len())
    .filter(c.count >= 15)
    .rename({"cited_case": "doc_id", "cited_case_right": "doc_id2"})
)

co_node, co_edge, co_G = DSNetwork.process(cocite_edges, directed=False)

co_node = (
    co_node
    .join(
        case.select(c.citation, c.case_name, c.term),
        left_on="id",
        right_on="citation",
        how="left"
    )
)
co_node

6. Draw the co-citation network in the same way as the last notebook, using `co_node`
and `co_edge`. Color the points by `eigen`.

Unlike the network in the last notebook, this one has several pieces. Two pairs of
cases sit off on their own: the death-penalty pair *Furman* and *Gregg*, and the
antitrust pair *Standard Oil* and *Socony*. Because the score is computed
separately for each piece, they both get an `eigen` of 1.

7. Filter `co_node` to the main component with `c.component == 1`. Then sort by
`eigen` from largest to smallest, keep the first 10 rows, and select `case_name`,
`term`, `eigen`, and `degree`.

The center of the co-citation network is a group of free-speech cases: *Cantwell*,
*Schneider*, *Whitney*, and *New York Times v. Sullivan*. Later opinions tend to
cite these cases together.

8. The block below is given. It builds a *directed* network from the citations among
the top cases and joins the case names onto the node table. The results are saved as
`di_node` and `di_edge`. Run it and look at `di_node`.

In [ ]:
edges = (
    cite
    .filter(c.citing_case.is_in(top), c.cited_case.is_in(top))
    .rename({"citing_case": "doc_id", "cited_case": "doc_id2"})
)

di_node, di_edge, di_G = DSNetwork.process(edges, directed=True)

di_node = (
    di_node
    .join(
        case.select(c.citation, c.case_name, c.term),
        left_on="id",
        right_on="citation",
        how="left"
    )
)
di_node

The directed network replaces `degree` with `degree_out`, the number of top cases
that a case cites, and `degree_in`, the number of top cases that cite it.

9. Sort `di_node` by `degree_in` from largest to smallest, keep the first 10 rows,
and select `case_name`, `term`, `degree_in`, and `degree_out`.

*Palko v. Connecticut* is cited by 18 of the other top cases. Most of the cases on
this list come from the 1920s and 1930s.

10. Draw a scatter plot of `di_node` with `degree_out` on the x-axis and `degree_in`
on the y-axis. Add a dashed line where the two are equal with
`.geom_abline(slope=1, intercept=0, linetype="dashed")`.

Many points sit far from the dashed line. Cases above the line are cited more than
they cite, and cases below it cite more than they are cited.

11. Draw a scatter plot of `di_node` with `term` on the x-axis and `degree_out` on
the y-axis.

Later cases cite more of the other top cases. A case can only cite cases that
came before it, so the oldest cases, such as *Marbury* and *McCulloch*, cite none
of the others.